<a href="https://colab.research.google.com/github/xyz111131/AI-Tools-for-Statistical-Research/blob/main/FashionMNIST_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets  ## domain-specific library, include datasets
from torchvision.transforms import ToTensor

# Download FashionMNIST data

In [ ]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(), # convert to tensor and scale to [0,1]
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 202kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.30MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 27.1MB/s]


The FashionMNIST dataset is a collection of 70,000 grayscale images of fashion articles, categorized into 10 classes. It serves as a direct drop-in replacement for the original MNIST dataset for benchmarking machine learning algorithms. Each image is 28x28 pixels.

In [ ]:
training_data

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [ ]:
test_data[0][1]

9

# Construct DataLoader

iterate over dataset, supports batching, shuffling etc.

In [ ]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

# Explanation of the tensor shape:
# N (Batch Size): 64 images are processed in one batch.
# C (Channels): 1, indicating a grayscale image (one color channel).
# H (Height): 28 pixels.
# W (Width): 28 pixels.

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


# Create MLP model

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)

print(model)

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


# Model training and testing

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3) # momentum=0, help escape local minima

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.111584  [   64/60000]
loss: 2.048632  [ 6464/60000]
loss: 2.075101  [12864/60000]
loss: 2.022936  [19264/60000]
loss: 2.036620  [25664/60000]
loss: 1.948272  [32064/60000]
loss: 1.986593  [38464/60000]
loss: 1.915759  [44864/60000]
loss: 1.840101  [51264/60000]
loss: 1.861359  [57664/60000]
Test Error: 
 Accuracy: 58.3%, Avg loss: 1.819274 

Epoch 2
-------------------------------
loss: 1.765616  [   64/60000]
loss: 1.843652  [ 6464/60000]
loss: 1.667212  [12864/60000]
loss: 1.760855  [19264/60000]
loss: 1.686585  [25664/60000]
loss: 1.726827  [32064/60000]
loss: 1.531142  [38464/60000]
loss: 1.527989  [44864/60000]
loss: 1.574627  [51264/60000]
loss: 1.425583  [57664/60000]
Test Error: 
 Accuracy: 62.9%, Avg loss: 1.464437 

Epoch 3
-------------------------------
loss: 1.526670  [   64/60000]
loss: 1.424377  [ 6464/60000]
loss: 1.382541  [12864/60000]
loss: 1.451046  [19264/60000]
loss: 1.425363  [25664/60000]
loss: 1.384454  [32064/600

# Saving Models

In [ ]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


# Loading Models

In [ ]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [ ]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x.unsqueeze(0)) # Unsqueeze to add batch dimension
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"\n')

Predicted: "Ankle boot", Actual: "Ankle boot"


--- Weight Shapes ---
linear_relu_stack.0.weight: torch.Size([512, 784])
linear_relu_stack.2.weight: torch.Size([512, 512])
linear_relu_stack.4.weight: torch.Size([10, 512])
